In [ ]:
import os
from pathlib import Path
import json
import sys
sys.path.append("/home/hdc/ljd1/github/conch")
from conch.conch import create_model_from_pretrained
from conch.downstream.zeroshot_path import zero_shot_classifier, run_zeroshot
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
import torch
import torch.nn.functional as F
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# display all jupyter output
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [3]:
root = Path('../').resolve()
os.chdir(root)

This notebook provides a complete example for performing zero-shot classification by ensembling multiple prompts and prompt templates. You can use this notebook to reproduce the zero-shot classification results on CRC100K (image size = 224 x 224). 

In [4]:
model_cfg = 'conch_ViT-B-16'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
checkpoint_path = '/data/ckpt/conch/pytorch_model.bin'
force_image_size = 224
model, preprocess = create_model_from_pretrained(model_cfg, checkpoint_path, device=device,
                                                 force_image_size=force_image_size)
_ = model.eval()

In [5]:
data_source = '/data/ljd/LLaVa/datasets/crc100k/CRC-VAL-HE-7K/'
dataset = ImageFolder(data_source, transform=preprocess)
dataloader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4)
if hasattr(dataloader.dataset, 'class_to_idx'):
     idx_to_class = {v:k for k,v in dataloader.dataset.class_to_idx.items()}
else:
     raise ValueError('Dataset does not have label_map attribute')
print("num samples: ", len(dataloader.dataset))
print(idx_to_class)

num samples:  7180
{0: 'ADI', 1: 'BACK', 2: 'DEB', 3: 'LYM', 4: 'MUC', 5: 'MUS', 6: 'NORM', 7: 'STR', 8: 'TUM'}


In [6]:
prompt_file = '/home/hdc/ljd1/github/conch/prompts/crc100k_prompts_all_per_class.json'
with open(prompt_file) as f:
    prompts = json.load(f)['0']
classnames = prompts['classnames']
templates = prompts['templates']
n_classes = len(classnames)
classnames_text = [classnames[str(idx_to_class[idx])] for idx in range(n_classes)]
for class_idx, classname in enumerate(classnames_text):
    print(f'{class_idx}: {classname}')

0: ['adipose', 'adipose tissue', 'adipocytes', 'fat', 'fat cells']
1: ['background', 'penmarking', 'empty space', 'background artifacts']
2: ['debris', 'colorectal adenocarcinoma debris and necrosis', 'necrosis', 'necrotic debris']
3: ['lymphocytes', 'lymphoid aggregate', 'immune cells', 'lymphoid infiltrate', 'inflammatory cells']
4: ['mucus', 'mucin', 'mucus pool', 'mucin pool']
5: ['smooth muscle', 'smooth muscle tissue', 'muscle', 'muscularis propria', 'muscularis mucosa']
6: ['normal colon mucosa', 'uninvolved colon mucosa', 'normal colonic mucosa', 'benign epithelium']
7: ['cancer-associated stroma', 'tumor-associated stroma', 'stromal cells', 'stromal tissue', 'stroma']
8: ['colorectal adenocarcinoma epithelium', 'colorectal adenocarcinoma', 'tumor', 'adenocarcinoma', 'malignant epithelium']


In [7]:
zeroshot_weights = zero_shot_classifier(model, classnames_text, templates, device=device)
print(zeroshot_weights.shape)

torch.Size([512, 9])


In [10]:
results, dump = run_zeroshot(model, zeroshot_weights, dataloader, device, 
                    dump_results=True, metrics=['acc', 'bacc', 'weighted_f1'])

100%|██████████| 113/113 [00:19<00:00,  5.66it/s]


In [11]:
for k, v in results.items():
    print(f'{k}: {v:.3f}')

acc: 0.803
bacc: 0.791
weighted_f1: 0.803
